# Workshop 4 Lab — Expert
### Computation 101 · IQ Biology Bootcamp 2026

This is the last lab, and it changes the job. You already know how to write bash, use git, and analyze
a real dataset in Python. Today you point an **AI assistant** at that same work — and learn the one skill
that makes AI safe for science: **you verify everything it hands you against something you already know.**

The data is the same **ZFP36L2 knockout** you analyzed in Workshop 3, and that's the whole trick: *you own
the answers.* When the assistant writes code, you don't hope it's right — you check its number against the
number you derived yourself. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your answer before running.

### First: open an AI assistant in another tab

This lab is **tool-agnostic** — any chat assistant works, and they're all free. Pick one:

- **ChatGPT** — chat.openai.com
- **Claude** — claude.ai
- **Google Gemini** — gemini.google.com (or aistudio.google.com)
- **Colab's own Gemini** — the **✦** button in the toolbar, or the Gemini side panel, right here in this window. Free for all Colab users, so there's nothing to install and no key to paste.

You'll copy a prompt from this notebook, paste it into your assistant, then bring the answer back here and
**run it against the ground truth.** No API keys, no setup — you are the bridge between the two tabs.

In [ ]:
# One-time setup. The line starting with ! is a REAL shell command in this Colab Linux
# machine — the same git you used on Fiji. It clones the course repo (only if it isn't
# here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Test first, then let the AI implement
The professional pattern with an AI assistant inverts how you've been working. You don't ask for code and
then wonder if it's right. You **write the test first** — encoding the answers you already own — and then ask
the assistant to write whatever implementation makes the test pass. The test is your safety net; the AI works
inside it. This is test-driven development, and AI makes it the natural way to work, not the disciplined luxury.

In [ ]:
# STEP 1 - the test, written first. These are the published results you reproduced across Workshops 3-4.
EXPECTED_UP = {'Lung': 71, 'Liver': 404, 'BM': 1343, 'Spleen': 573, 'Ovary': 291, 'Kidney': 53}
UNIVERSAL   = {'ENSMUSG00000091694'}   # the one gene up in all six tissues (Apol11b)
N_GE4       = 16                        # genes up in >= 4 tissues

def check(analyze):
    up, universal, n_ge4 = analyze(DATA)
    assert up == EXPECTED_UP,   f'up-counts wrong: {up}'
    assert universal == UNIVERSAL, f'intersection wrong: {universal}'
    assert n_ge4 == N_GE4,      f'>=4-tissue count wrong: {n_ge4}'
    print('ALL TESTS PASS ✓  — the implementation reproduces every published landmark')

## Step 2 · Now ask the assistant to satisfy the test
Hand your assistant the contract, not the solution:

> Write `analyze(data_dir)` returning a tuple `(up_counts, universal, n_ge4)`. For tissues Lung, Liver, BM,
> Spleen, Ovary, Kidney it reads `{data_dir}/de/{tissue}_DE.csv` (gene IDs in column 0). `up_counts` is a dict
> tissue → number of genes with log2FoldChange > 1 and padj < 0.05. `universal` is the set of gene IDs up in
> **all six** tissues. `n_ge4` is how many gene IDs are up in **four or more** tissues.

Paste its answer below. Then run `check(analyze)` — you'll know instantly whether to trust it.

In [ ]:
from collections import Counter
def analyze(data_dir):
    tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
    up_sets, up_counts = {}, {}
    for t in tissues:
        df = pd.read_csv(os.path.join(data_dir, 'de', f'{t}_DE.csv'), index_col=0)
        hits = set(df[(df['log2FoldChange'] > 1) & (df['padj'] < 0.05)].index)
        up_sets[t] = hits
        up_counts[t] = len(hits)
    universal = ...                        # in ALL six sets
    counts = Counter(g for s in up_sets.values() for g in s)
    n_ge4 = ...                            # how many genes appear in >= 4 sets
    return up_counts, universal, n_ge4

check(analyze)

If a single number had been off — a `>` that should have been `>=`, a `pvalue` for a `padj` — the test would
have told you on the spot, with the exact mismatch. That is the entire value of writing it first: the AI can
be as fast and as confident as it likes, because it cannot get anything past a test you wrote from ground truth.

## Step 3 · Honest attribution
One rule closes the bootcamp. When AI helped you produce a result, you **say so** — plainly, in the repo, so
the next person knows exactly what the machine did and what you verified. It costs one file and it is simply
how honest computational work is done now. Edit this note to match what you actually did, then write it out.

In [ ]:
note = '''# How AI was used in this analysis

Tool: <the assistant you used>

What I asked it to write:
- analyze(data_dir): per-tissue up-regulated counts, the six-way intersection, and the >=4-tissue count.

How I verified it:
- Wrote the test (check()) FIRST from the published landmarks (per-tissue up-counts, Apol11b as the
  universal gene, 16 genes up in >=4 tissues), then required the AI's implementation to pass it.

What I changed by hand:
- <anything you edited, rejected, or rewrote>
'''
with open('AI_NOTES.md', 'w') as f:
    f.write(note)
print(open('AI_NOTES.md').read())

## The capstone — commit it, with its provenance
You've done this loop before, in Workshop 2. Now the artifact is an AI-assisted, test-backed analysis that
carries an honest note about how it was made. Download this notebook (**File → Download → .ipynb**), move it
to Fiji next to `AI_NOTES.md`, and run the same push you already know:

```bash
cd ~/zfp36l2-analysis
git add analysis.ipynb AI_NOTES.md
git commit -m 'Add AI-assisted tissue-count analysis, verified against published landmarks'
git push
```

That commit is the whole bootcamp in one object: real data, under version control, analyzed in Python,
accelerated by AI, and **documented honestly**. bash → git → Python → AI, closed.

## One more thing

You just used AI the way a scientist has to — as a fast, confident junior colleague whose every claim you
check before you trust it. And you *could* check it, because the ground truth was real: a published study,
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program a year ago, sitting where you are now.
Across four workshops you took one real dataset from a raw shell prompt to a version-controlled, Python-analyzed,
AI-assisted result: **bash → git → Python → AI**, every step on the same science. The assistant made you faster.
Owning the answer is what kept you honest. That's the job now — and you can already do it.